In [ ]:
# Recommended configurations
recommendations = {
    "Small Model (Fast Inference)": {
        "image_size": 64,
        "patch_size": 8,
        "num_patches": 64,
        "embed_dim": 384,
        "depth": 6,
        "num_heads": 6,
        "mlp_ratio": 4,
        "position_embedding": "2d",
        "patch_embedding": "standard"
    },
    "Base Model (Balanced)": {
        "image_size": 128,
        "patch_size": 16,
        "num_patches": 64,
        "embed_dim": 768,
        "depth": 12,
        "num_heads": 12,
        "mlp_ratio": 4,
        "position_embedding": "learnable",
        "patch_embedding": "hybrid"
    },
    "Large Model (High Accuracy)": {
        "image_size": 224,
        "patch_size": 16,
        "num_patches": 196,
        "embed_dim": 1024,
        "depth": 24,
        "num_heads": 16,
        "mlp_ratio": 4,
        "position_embedding": "2d",
        "patch_embedding": "adaptive"
    }
}

# Display recommendations
for model_type, config in recommendations.items():
    print(f"\n{model_type}:")
    print("-" * 40)
    for key, value in config.items():
        print(f"{key:20s}: {value}")
    
    # Calculate model statistics
    params = (config['embed_dim'] * config['num_patches'] +  # Patch embedding
             config['embed_dim'] * config['embed_dim'] * 3 * config['depth'] +  # Attention
             config['embed_dim'] * config['embed_dim'] * config['mlp_ratio'] * 2 * config['depth'])  # MLP
    
    print(f"{'Approx. Parameters':20s}: {params/1e6:.1f}M")
    print(f"{'FLOPs per forward':20s}: {params * 2 / 1e9:.2f}G")

# Key insights summary
print("\n\nKey Insights for Packet Data:")
print("=" * 50)
print("1. Patch Size Selection:")
print("   - 8x8: Captures fine-grained byte patterns, good for small packets")
print("   - 16x16: Best balance for most use cases")
print("   - 32x32: Suitable for high-level pattern detection")
print("\n2. Image Size Considerations:")
print("   - 64x64: Covers first 4KB of packet data")
print("   - 128x128: Covers ~16KB, suitable for most packets")
print("   - 224x224: Standard ViT size, may require significant padding")
print("\n3. Position Embedding Strategy:")
print("   - 2D embeddings work well for spatial byte relationships")
print("   - Learnable embeddings offer flexibility")
print("   - Byte-order aware embeddings can leverage packet structure")
print("\n4. Patch Diversity:")
print("   - Attack packets show higher entropy in patches")
print("   - Benign traffic has more uniform patch distributions")
print("   - Adaptive sampling can focus on high-entropy regions")

## 7. Recommendations for Packet-ViT Architecture

Based on our analysis, here are the optimal configurations for Vision Transformers on packet data:

In [ ]:
class PositionEmbedding(nn.Module):
    """
    Various position embedding strategies for packet image patches.
    """
    
    def __init__(self, 
                 num_patches: int,
                 embed_dim: int,
                 embedding_type: str = 'learnable'):
        """
        Args:
            num_patches: Total number of patches
            embed_dim: Embedding dimension
            embedding_type: Type of position embedding ('learnable', 'sinusoidal', '2d')
        """
        super().__init__()
        self.num_patches = num_patches
        self.embed_dim = embed_dim
        self.embedding_type = embedding_type
        
        if embedding_type == 'learnable':
            self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
            nn.init.trunc_normal_(self.pos_embed, std=0.02)
            
        elif embedding_type == 'sinusoidal':
            self.register_buffer('pos_embed', self._create_sinusoidal_embedding())
            
        elif embedding_type == '2d':
            # Assume square grid of patches
            grid_size = int(np.sqrt(num_patches))
            self.register_buffer('pos_embed', self._create_2d_embedding(grid_size))
            
    def _create_sinusoidal_embedding(self) -> torch.Tensor:
        """Create sinusoidal position embeddings."""
        position = torch.arange(self.num_patches).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, self.embed_dim, 2) * 
                           -(np.log(10000.0) / self.embed_dim))
        
        pos_embed = torch.zeros(1, self.num_patches, self.embed_dim)
        pos_embed[0, :, 0::2] = torch.sin(position * div_term)
        pos_embed[0, :, 1::2] = torch.cos(position * div_term)
        
        return pos_embed
    
    def _create_2d_embedding(self, grid_size: int) -> torch.Tensor:
        """Create 2D position embeddings considering spatial structure."""
        pos_embed = torch.zeros(1, grid_size * grid_size, self.embed_dim)
        
        # Create 2D positional grid
        y_embed = torch.arange(grid_size).unsqueeze(1).repeat(1, grid_size).flatten()
        x_embed = torch.arange(grid_size).repeat(grid_size)
        
        # Encode positions
        div_term = torch.exp(torch.arange(0, self.embed_dim // 2, 2) * 
                           -(np.log(10000.0) / (self.embed_dim // 2)))
        
        # Y-axis encoding
        pos_embed[0, :, 0:self.embed_dim//2:2] = torch.sin(y_embed.unsqueeze(1) * div_term)
        pos_embed[0, :, 1:self.embed_dim//2:2] = torch.cos(y_embed.unsqueeze(1) * div_term)
        
        # X-axis encoding  
        pos_embed[0, :, self.embed_dim//2::2] = torch.sin(x_embed.unsqueeze(1) * div_term)
        pos_embed[0, :, self.embed_dim//2+1::2] = torch.cos(x_embed.unsqueeze(1) * div_term)
        
        return pos_embed
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add position embeddings to input."""
        return x + self.pos_embed


class ByteOrderAwarePositionEmbedding(nn.Module):
    """
    Position embedding that considers the original byte order in packets.
    Useful when patches might represent contiguous byte sequences.
    """
    
    def __init__(self,
                 num_patches: int,
                 embed_dim: int,
                 patch_size: int = 16,
                 image_size: int = 224):
        super().__init__()
        self.num_patches = num_patches
        self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.image_size = image_size
        
        # Learnable embeddings for different position ranges
        self.header_embed = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.payload_embed = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.position_scale = nn.Parameter(torch.ones(1, 1, embed_dim))
        
        # Base position embedding
        self.base_pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        
        nn.init.trunc_normal_(self.header_embed, std=0.02)
        nn.init.trunc_normal_(self.payload_embed, std=0.02)
        nn.init.trunc_normal_(self.base_pos_embed, std=0.02)
        
    def forward(self, x: torch.Tensor, packet_lengths: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Add position embeddings with byte-order awareness.
        
        Args:
            x: Patch embeddings (B, num_patches, embed_dim)
            packet_lengths: Original packet lengths for each sample in batch
        """
        B = x.shape[0]
        
        # Base position embedding
        pos_embed = self.base_pos_embed.expand(B, -1, -1)
        
        # Add byte-order aware components
        patches_per_row = self.image_size // self.patch_size
        
        for b in range(B):
            for i in range(self.num_patches):
                # Calculate which bytes this patch represents
                row = i // patches_per_row
                col = i % patches_per_row
                byte_start = row * self.image_size + col * self.patch_size
                
                # Add header/payload distinction (first ~40 bytes are typically headers)
                if byte_start < 40:
                    pos_embed[b, i] += self.header_embed
                else:
                    pos_embed[b, i] += self.payload_embed
                    
        return x + pos_embed * self.position_scale

# Visualize different position embedding strategies
def visualize_position_embeddings():
    """Visualize different position embedding patterns."""
    num_patches = 196  # 14x14 patches for 224x224 image with 16x16 patches
    embed_dim = 768
    
    # Create different position embeddings
    learnable_pe = PositionEmbedding(num_patches, embed_dim, 'learnable')
    sinusoidal_pe = PositionEmbedding(num_patches, embed_dim, 'sinusoidal')
    pe_2d = PositionEmbedding(num_patches, embed_dim, '2d')
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Visualize first few dimensions of each embedding
    dims_to_show = 128
    
    # Learnable (random initialization)
    ax = axes[0]
    im = ax.imshow(learnable_pe.pos_embed[0, :, :dims_to_show].T.detach().numpy(), 
                   aspect='auto', cmap='coolwarm')
    ax.set_title('Learnable Position Embedding')
    ax.set_xlabel('Patch Index')
    ax.set_ylabel('Embedding Dimension')
    
    # Sinusoidal
    ax = axes[1]
    im = ax.imshow(sinusoidal_pe.pos_embed[0, :, :dims_to_show].T.numpy(), 
                   aspect='auto', cmap='coolwarm')
    ax.set_title('Sinusoidal Position Embedding')
    ax.set_xlabel('Patch Index')
    
    # 2D
    ax = axes[2]
    im = ax.imshow(pe_2d.pos_embed[0, :, :dims_to_show].T.numpy(), 
                   aspect='auto', cmap='coolwarm')
    ax.set_title('2D Position Embedding')
    ax.set_xlabel('Patch Index')
    
    plt.suptitle('Position Embedding Strategies Visualization', fontsize=16)
    plt.tight_layout()
    plt.colorbar(im, ax=axes.ravel().tolist(), fraction=0.046, pad=0.04)
    plt.show()

visualize_position_embeddings()

## 6. Position Embeddings for Packet Data

Position embeddings are crucial for ViT. For packet data, we can leverage the sequential nature of bytes.

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Patch Embedding layer for Vision Transformer.
    Converts packet images into patch embeddings.
    """
    
    def __init__(self, 
                 image_size: int = 224,
                 patch_size: int = 16,
                 in_channels: int = 1,
                 embed_dim: int = 768,
                 norm_layer: Optional[nn.Module] = None):
        """
        Args:
            image_size: Size of input square image
            patch_size: Size of each patch
            in_channels: Number of input channels (1 for grayscale)
            embed_dim: Dimension of patch embeddings
            norm_layer: Optional normalization layer
        """
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        
        # Convolutional projection - acts as both patch extraction and linear projection
        self.proj = nn.Conv2d(in_channels, embed_dim, 
                            kernel_size=patch_size, stride=patch_size)
        
        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape (B, C, H, W)
            
        Returns:
            Patch embeddings of shape (B, num_patches, embed_dim)
        """
        B, C, H, W = x.shape
        assert H == W == self.image_size, \
            f"Input image size ({H}x{W}) doesn't match expected size ({self.image_size}x{self.image_size})"
        
        # Extract and project patches: (B, C, H, W) -> (B, embed_dim, n_patches_h, n_patches_w)
        x = self.proj(x)
        
        # Flatten patches: (B, embed_dim, n_h, n_w) -> (B, embed_dim, num_patches)
        x = x.flatten(2)
        
        # Transpose: (B, embed_dim, num_patches) -> (B, num_patches, embed_dim)
        x = x.transpose(1, 2)
        
        # Apply normalization
        x = self.norm(x)
        
        return x


class HybridPatchEmbedding(nn.Module):
    """
    Hybrid CNN-Transformer patch embedding.
    Uses CNN features before patch extraction for richer representations.
    """
    
    def __init__(self,
                 image_size: int = 224,
                 patch_size: int = 16,
                 in_channels: int = 1,
                 embed_dim: int = 768,
                 backbone: str = 'simple'):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        
        # Simple CNN backbone for feature extraction
        if backbone == 'simple':
            self.backbone = nn.Sequential(
                nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
                nn.Conv2d(32, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
            )
            feature_dim = 64
        else:
            raise ValueError(f"Unknown backbone: {backbone}")
        
        # Patch projection
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(feature_dim, embed_dim,
                            kernel_size=patch_size, stride=patch_size)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Extract CNN features then create patch embeddings."""
        # Extract CNN features
        x = self.backbone(x)
        
        # Project to patch embeddings
        x = self.proj(x)
        
        # Reshape: (B, embed_dim, n_h, n_w) -> (B, num_patches, embed_dim)
        x = x.flatten(2).transpose(1, 2)
        
        return x


class AdaptivePatchEmbedding(nn.Module):
    """
    Adaptive patch embedding that can handle variable patch sizes
    or importance-based sampling.
    """
    
    def __init__(self,
                 image_size: int = 224,
                 patch_sizes: List[int] = [8, 16, 32],
                 in_channels: int = 1,
                 embed_dim: int = 768):
        super().__init__()
        self.image_size = image_size
        self.patch_sizes = patch_sizes
        
        # Multiple patch extractors for different scales
        self.projections = nn.ModuleList([
            nn.Conv2d(in_channels, embed_dim // len(patch_sizes),
                     kernel_size=ps, stride=ps)
            for ps in patch_sizes
        ])
        
        # Learnable scale embedding
        self.scale_embeddings = nn.ParameterList([
            nn.Parameter(torch.randn(1, 1, embed_dim // len(patch_sizes)))
            for _ in patch_sizes
        ])
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Extract multi-scale patches and combine."""
        B = x.shape[0]
        all_patches = []
        
        for proj, scale_emb, patch_size in zip(self.projections, 
                                              self.scale_embeddings, 
                                              self.patch_sizes):
            # Extract patches at this scale
            patches = proj(x)  # (B, C', H', W')
            patches = patches.flatten(2).transpose(1, 2)  # (B, num_patches, C')
            
            # Add scale embedding
            patches = patches + scale_emb
            all_patches.append(patches)
        
        # Concatenate all scales
        x = torch.cat(all_patches, dim=2)  # (B, total_patches, embed_dim)
        
        return x

# Test implementations
def test_patch_embeddings():
    """Test different patch embedding implementations."""
    B, C, H, W = 4, 1, 224, 224
    x = torch.randn(B, C, H, W)
    
    # Standard patch embedding
    patch_embed = PatchEmbedding(image_size=H, patch_size=16, embed_dim=768)
    out = patch_embed(x)
    print(f"Standard Patch Embedding output shape: {out.shape}")
    
    # Hybrid patch embedding
    hybrid_embed = HybridPatchEmbedding(image_size=H, patch_size=16, embed_dim=768)
    out = hybrid_embed(x)
    print(f"Hybrid Patch Embedding output shape: {out.shape}")
    
    # Adaptive patch embedding
    adaptive_embed = AdaptivePatchEmbedding(image_size=H, patch_sizes=[8, 16, 32], embed_dim=768)
    out = adaptive_embed(x)
    print(f"Adaptive Patch Embedding output shape: {out.shape}")

test_patch_embeddings()

## 5. PyTorch Implementation for ViT Integration

In [ ]:
def analyze_patch_diversity(patches: np.ndarray) -> Dict:
    """Analyze diversity of extracted patches."""
    # Flatten patches for analysis
    flat_patches = patches.reshape(len(patches), -1)
    
    # Calculate statistics
    stats = {
        'mean_intensity': np.mean(flat_patches, axis=1),
        'std_intensity': np.std(flat_patches, axis=1),
        'entropy': []
    }
    
    # Calculate entropy for each patch
    for patch in flat_patches:
        hist, _ = np.histogram(patch, bins=256, range=(0, 256))
        hist = hist / hist.sum()
        hist = hist[hist > 0]  # Remove zeros
        entropy = -np.sum(hist * np.log2(hist))
        stats['entropy'].append(entropy)
    
    stats['entropy'] = np.array(stats['entropy'])
    
    # Calculate diversity metrics
    diversity = {
        'mean_diversity': np.std(stats['mean_intensity']),
        'entropy_diversity': np.std(stats['entropy']),
        'low_entropy_ratio': np.sum(stats['entropy'] < 2) / len(patches),
        'high_entropy_ratio': np.sum(stats['entropy'] > 6) / len(patches)
    }
    
    return stats, diversity

# Analyze patches for different traffic types
encoder = PacketImageEncoder(image_size=(128, 128))
extractor = PatchExtractor(patch_size=16)

# Sample multiple packets per class
n_samples = 20
diversity_results = []

for label in range(6):  # 0 is benign, 1-5 are attacks
    if label == 0:
        packets = cic_data[cic_data['label'] == label][byte_columns].values[:n_samples]
        label_name = 'Benign'
    else:
        packets = cic_data[cic_data['label'] == label][byte_columns].values[:n_samples]
        label_name = f'Attack-{label}'
    
    all_entropies = []
    all_means = []
    
    for packet in packets:
        # Encode and extract patches
        img = encoder.encode_sequential(packet)
        patches, _ = extractor.extract_patches(img)
        stats, diversity = analyze_patch_diversity(patches)
        
        all_entropies.extend(stats['entropy'])
        all_means.extend(stats['mean_intensity'])
    
    diversity_results.append({
        'label': label_name,
        'avg_entropy': np.mean(all_entropies),
        'entropy_std': np.std(all_entropies),
        'avg_intensity': np.mean(all_means),
        'intensity_std': np.std(all_means)
    })

# Visualize diversity analysis
div_df = pd.DataFrame(diversity_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Entropy comparison
ax = axes[0]
ax.bar(div_df['label'], div_df['avg_entropy'], yerr=div_df['entropy_std'], capsize=5)
ax.set_title('Average Patch Entropy by Traffic Type')
ax.set_ylabel('Entropy (bits)')
ax.tick_params(axis='x', rotation=45)

# Intensity comparison
ax = axes[1]
ax.bar(div_df['label'], div_df['avg_intensity'], yerr=div_df['intensity_std'], capsize=5)
ax.set_title('Average Patch Intensity by Traffic Type')
ax.set_ylabel('Mean Pixel Value')
ax.tick_params(axis='x', rotation=45)

plt.suptitle('Patch Diversity Analysis Across Traffic Types', fontsize=16)
plt.tight_layout()
plt.show()

print("\nPatch Diversity Metrics:")
print(div_df.round(3))

## 4. Patch Diversity Analysis

Let's analyze how diverse the patches are for different traffic types to understand if smaller patches capture meaningful patterns.

In [ ]:
# Compare different patch sizes on 64x64 image
patch_sizes = [8, 16, 32]
fig, axes = plt.subplots(1, len(patch_sizes), figsize=(15, 5))

for idx, patch_size in enumerate(patch_sizes):
    extractor = PatchExtractor(patch_size=patch_size)
    patches, info = extractor.extract_patches(img_64)
    
    ax = axes[idx]
    ax.imshow(img_64, cmap='gray', vmin=0, vmax=255)
    
    # Draw patch grid
    for y, x in info['positions']:
        rect = Rectangle((x, y), patch_size, patch_size, 
                       linewidth=1, edgecolor='red', facecolor='none', alpha=0.7)
        ax.add_patch(rect)
    
    ax.set_title(f'Patch Size: {patch_size}x{patch_size}\n{info["num_patches"]} patches')
    ax.axis('off')

plt.suptitle('Patch Size Comparison on 64x64 Image', fontsize=16)
plt.tight_layout()
plt.savefig(figures_dir / 'patch_size_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate sequence lengths for different configurations
configs = []
for img_size in [64, 128, 224]:
    for patch_size in [8, 16, 32]:
        if img_size % patch_size == 0:
            seq_len = (img_size // patch_size) ** 2
            configs.append({
                'image_size': img_size,
                'patch_size': patch_size,
                'sequence_length': seq_len,
                'patches_per_dim': img_size // patch_size
            })

config_df = pd.DataFrame(configs)
print("\nViT Configuration Options:")
print(config_df)

## 3. Compare Different Patch Sizes

In [ ]:
class PatchExtractor:
    """
    Advanced patch extraction for Vision Transformers with packet data.
    """
    
    def __init__(self, patch_size: int = 16, overlap: int = 0):
        """
        Initialize patch extractor.
        
        Args:
            patch_size: Size of each square patch
            overlap: Number of pixels to overlap between patches
        """
        self.patch_size = patch_size
        self.overlap = overlap
        self.stride = patch_size - overlap
        
    def extract_patches(self, image: np.ndarray) -> Tuple[np.ndarray, Dict]:
        """
        Extract patches from an image.
        
        Args:
            image: 2D numpy array representing the image
            
        Returns:
            patches: Array of shape (num_patches, patch_size, patch_size)
            info: Dictionary with patch information
        """
        h, w = image.shape
        
        # Calculate number of patches
        n_patches_h = (h - self.patch_size) // self.stride + 1
        n_patches_w = (w - self.patch_size) // self.stride + 1
        
        patches = []
        positions = []
        
        for i in range(n_patches_h):
            for j in range(n_patches_w):
                # Calculate patch position
                y = i * self.stride
                x = j * self.stride
                
                # Extract patch
                patch = image[y:y+self.patch_size, x:x+self.patch_size]
                patches.append(patch)
                positions.append((y, x))
        
        patches = np.array(patches)
        
        info = {
            'num_patches': len(patches),
            'patches_per_row': n_patches_w,
            'patches_per_col': n_patches_h,
            'positions': positions,
            'coverage_ratio': (patches.size / image.size)
        }
        
        return patches, info
    
    def extract_patches_adaptive(self, image: np.ndarray, 
                               importance_map: Optional[np.ndarray] = None) -> Tuple[np.ndarray, Dict]:
        """
        Extract patches with adaptive sampling based on importance.
        
        Args:
            image: 2D numpy array
            importance_map: Optional importance weights for each pixel
            
        Returns:
            patches: Extracted patches
            info: Extraction information
        """
        if importance_map is None:
            # Use gradient magnitude as default importance
            importance_map = self._compute_gradient_importance(image)
        
        h, w = image.shape
        
        # Fixed grid extraction first
        patches, info = self.extract_patches(image)
        
        # Additional patches from high-importance regions
        threshold = np.percentile(importance_map, 90)  # Top 10% important regions
        important_regions = np.where(importance_map > threshold)
        
        additional_patches = []
        additional_positions = []
        
        # Sample additional patches from important regions
        for idx in range(0, len(important_regions[0]), self.patch_size // 2):
            y, x = important_regions[0][idx], important_regions[1][idx]
            
            # Ensure patch fits within image
            if y + self.patch_size <= h and x + self.patch_size <= w:
                patch = image[y:y+self.patch_size, x:x+self.patch_size]
                additional_patches.append(patch)
                additional_positions.append((y, x))
        
        if additional_patches:
            all_patches = np.concatenate([patches, np.array(additional_patches)])
            info['positions'].extend(additional_positions)
            info['num_adaptive_patches'] = len(additional_patches)
        else:
            all_patches = patches
            info['num_adaptive_patches'] = 0
            
        info['num_patches'] = len(all_patches)
        
        return all_patches, info
    
    def _compute_gradient_importance(self, image: np.ndarray) -> np.ndarray:
        """Compute importance map based on gradient magnitude."""
        # Sobel gradients
        grad_x = np.abs(np.diff(image.astype(np.float32), axis=1, prepend=image[:, :1]))
        grad_y = np.abs(np.diff(image.astype(np.float32), axis=0, prepend=image[:1, :]))
        
        # Gradient magnitude
        grad_mag = np.sqrt(grad_x**2 + grad_y[:-1, :]**2)
        
        # Gaussian smoothing
        from scipy.ndimage import gaussian_filter
        importance = gaussian_filter(grad_mag, sigma=2)
        
        return importance
    
    def visualize_patches(self, image: np.ndarray, patches: np.ndarray, 
                         info: Dict, title: str = "Patch Extraction"):
        """Visualize patch extraction on the original image."""
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # Original image with patch boundaries
        ax = axes[0]
        ax.imshow(image, cmap='gray', vmin=0, vmax=255)
        ax.set_title(f'Original Image with Patch Grid\n({info["num_patches"]} patches)')
        
        # Draw patch boundaries
        for y, x in info['positions']:
            rect = Rectangle((x, y), self.patch_size, self.patch_size, 
                           linewidth=1, edgecolor='red', facecolor='none', alpha=0.5)
            ax.add_patch(rect)
        
        ax.axis('off')
        
        # Show some extracted patches
        ax = axes[1]
        n_show = min(16, len(patches))
        grid_size = int(np.sqrt(n_show))
        
        for i in range(n_show):
            plt.subplot(1, 2, 2)
            ax = plt.subplot(2, grid_size*2, grid_size*2 + i + 1)
            ax.imshow(patches[i], cmap='gray', vmin=0, vmax=255)
            ax.axis('off')
            ax.set_title(f'P{i}', fontsize=8)
        
        plt.suptitle(title)
        plt.tight_layout()
        return fig

# Create patch extractor
extractor = PatchExtractor(patch_size=16)

## 2. Patch Extraction Implementation

In [ ]:
# Load sample data
cic_data = pd.read_csv(raw_data_dir / 'payload_byte' / 'cic_ids2017_sample.csv')
byte_columns = [f'byte_{i}' for i in range(1500)]

# Get sample packets
sample_benign = cic_data[cic_data['label'] == 0][byte_columns].values[0]
sample_attack = cic_data[cic_data['label'] != 0][byte_columns].values[0]

# Create packet images with different sizes
encoder_64 = PacketImageEncoder(image_size=(64, 64))
encoder_128 = PacketImageEncoder(image_size=(128, 128))
encoder_224 = PacketImageEncoder(image_size=(224, 224))

# Generate images
img_64 = encoder_64.encode_sequential(sample_benign)
img_128 = encoder_128.encode_sequential(sample_benign)
img_224 = encoder_224.encode_sequential(sample_benign)

print(f"Image sizes: {img_64.shape}, {img_128.shape}, {img_224.shape}")

## 1. Understanding Vision Transformer Patch Requirements

Vision Transformers divide input images into non-overlapping patches and treat them as sequences. Key considerations for packet data:

1. **Patch Size**: Typical sizes are 8x8, 16x16, or 32x32
2. **Image Size**: Must be divisible by patch size
3. **Sequence Length**: Number of patches = (H/P) × (W/P) where P is patch size
4. **Information Density**: Smaller patches capture finer details but increase sequence length

In [ ]:
# Setup project paths
notebook_path = Path().resolve()
project_root = notebook_path.parent

# Add project root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import our packet encoder
from src.data.packet_to_image import PacketImageEncoder, create_patch_embeddings

# Define paths
data_dir = project_root / 'data'
raw_data_dir = data_dir / 'raw'
figures_dir = project_root / 'notebooks' / 'figures'

print(f"Project root: {project_root}")
print(f"Data directory: {data_dir}")

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, List, Optional, Dict
import warnings
warnings.filterwarnings('ignore')

# Visualization
from matplotlib.patches import Rectangle
from matplotlib.patches import Patch
import matplotlib.cm as cm

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')

# Configure notebook display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 05 - Patch Extraction Strategies for Vision Transformers

This notebook explores different patch extraction strategies for Vision Transformers (ViT) applied to packet image data. We'll analyze patch sizes, overlapping strategies, and adaptive methods to optimize for malware detection.